**MGMT298D: Science and Strategy of AI**
# Week 2: XGBoost & Hyperparameter Tuning

#### This notebook introduces `xgboost`, a powerful gradient-boosted tree library, and walks through a systematic hyperparameter tuning process. We load a car pricing dataset, encode categorical features, establish a baseline, then search over learning rate, number of estimators, and tree depth to find the best model.

# 1 Setup & Imports

#### We import `pandas` for data handling, `numpy` for numerical operations, `sklearn` for encoding, cross-validation, and error metrics, and `xgboost` for gradient-boosted regression.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# 1.1 Load Data

#### We read the Range Rover dataset from a CSV and inspect its shape, columns, and first few rows.

In [ ]:
url = 'https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/range_rover.csv'
df = pd.read_csv(url)
print(f"{len(df)} rows, {df.shape[1]} columns")

pd.set_option('display.max_columns', None)
df.head()

# 1.2 Encode Features

#### Categorical columns like trim, state, and color need to be converted to numbers before `xgboost` can use them. We apply `LabelEncoder` from `sklearn` to each one, then define our feature set and target.

In [ ]:
target_col = 'sellingprice' if 'sellingprice' in df.columns else 'price'

# Label-encode categorical features
le_trim = LabelEncoder()
le_state = LabelEncoder()
le_color = LabelEncoder()

df['trim_enc'] = le_trim.fit_transform(df['trim'])
df['state_enc'] = le_state.fit_transform(df['state'])
df['color_enc'] = le_color.fit_transform(df['color'])

feature_cols = ['year', 'mileage', 'trim_enc', 'state_enc', 'color_enc']
X = df[feature_cols].copy()
y = df[target_col].copy()

print(f"Features: {feature_cols}")
print(f"Target: {target_col}")
print(f"\nTarget statistics:")
print(y.describe())

---
# 2 Baseline Model

#### We train an `XGBRegressor` with default hyperparameters and evaluate it with 5-fold cross-validation. This gives us a baseline MAE to improve upon.

In [ ]:
xgb_baseline = XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)
baseline_scores = cross_val_score(xgb_baseline, X, y, cv=5, scoring='neg_mean_absolute_error')
baseline_mae = -baseline_scores.mean()

print(f"Baseline XGBRegressor (5-fold CV)")
print(f"  MAE: ${baseline_mae:,.2f}")
print(f"  Std: ${baseline_scores.std():,.2f}")
print(f"  Default learning_rate: {xgb_baseline.learning_rate}")
print(f"  Default n_estimators: {xgb_baseline.n_estimators}")
print(f"  Default max_depth: {xgb_baseline.max_depth}")

---
# 3 Hyperparameter Search

#### We sweep one hyperparameter at a time — learning rate, number of estimators, and max depth — evaluating each with 5-fold CV to see how it affects MAE.

#### Learning rate controls how much each new tree corrects the previous ones. Too low is slow, too high overshoots.

In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.2, 0.3]
lr_results = []

for lr in learning_rates:
    xgb = XGBRegressor(learning_rate=lr, random_state=42, n_jobs=-1, verbosity=0)
    scores = cross_val_score(xgb, X, y, cv=5, scoring='neg_mean_absolute_error')
    mae = -scores.mean()
    lr_results.append({'learning_rate': lr, 'MAE': mae})

lr_df = pd.DataFrame(lr_results)
print(lr_df.to_string(index=False))

In [ ]:
plt.plot(lr_df['learning_rate'], lr_df['MAE'], 'o-')
plt.xlabel('Learning Rate')
plt.ylabel('MAE ($)')
plt.title('Learning Rate vs MAE')
plt.show()

#### Number of estimators is how many trees are built sequentially. More trees can improve accuracy but eventually overfit.

In [ ]:
plt.plot(ne_df['n_estimators'], ne_df['MAE'], 'o-')
plt.xlabel('Number of Estimators')
plt.ylabel('MAE ($)')
plt.title('N Estimators vs MAE')
plt.show()

In [ ]:
n_estimators_list = [25, 50, 75, 100, 150, 200]
ne_results = []

for ne in n_estimators_list:
    xgb = XGBRegressor(n_estimators=ne, random_state=42, n_jobs=-1, verbosity=0)
    scores = cross_val_score(xgb, X, y, cv=5, scoring='neg_mean_absolute_error')
    mae = -scores.mean()
    ne_results.append({'n_estimators': ne, 'MAE': mae})

ne_df = pd.DataFrame(ne_results)
print(ne_df.to_string(index=False))

In [ ]:
plt.plot(md_df['max_depth'], md_df['MAE'], 'o-')
plt.xlabel('Max Depth')
plt.ylabel('MAE ($)')
plt.title('Max Depth vs MAE')
plt.show()

#### Max depth limits how deep each individual tree can grow. Deeper trees capture more complex patterns but are more prone to overfitting.

In [ ]:
plt.imshow(grid_results, aspect='auto')
plt.xticks(range(len(n_estimators_grid)), n_estimators_grid)
plt.yticks(range(len(learning_rates_grid)), learning_rates_grid)
plt.xlabel('N Estimators')
plt.ylabel('Learning Rate')
plt.colorbar(label='MAE ($)')
plt.title('Grid Search: Learning Rate × N Estimators')
plt.show()

In [ ]:
max_depths = [2, 3, 4, 5, 6, 8]
md_results = []

for md in max_depths:
    xgb = XGBRegressor(max_depth=md, random_state=42, n_jobs=-1, verbosity=0)
    scores = cross_val_score(xgb, X, y, cv=5, scoring='neg_mean_absolute_error')
    mae = -scores.mean()
    md_results.append({'max_depth': md, 'MAE': mae})

md_df = pd.DataFrame(md_results)
print(md_df.to_string(index=False))

In [ ]:
sorted_idx = np.argsort(importances)
plt.barh([feature_cols[i] for i in sorted_idx], importances[sorted_idx])
plt.xlabel('Importance')
plt.title('Feature Importances')
plt.show()

---
# 4 2D Grid Search

#### We now search over learning rate and number of estimators jointly. This captures interactions between the two hyperparameters that single-parameter sweeps miss.

In [ ]:
learning_rates_grid = [0.01, 0.05, 0.1, 0.2, 0.3]
n_estimators_grid = [25, 50, 75, 100, 150, 200]

grid_results = np.zeros((len(learning_rates_grid), len(n_estimators_grid)))

for i, lr in enumerate(learning_rates_grid):
    for j, ne in enumerate(n_estimators_grid):
        xgb = XGBRegressor(learning_rate=lr, n_estimators=ne,
                          random_state=42, n_jobs=-1, verbosity=0)
        scores = cross_val_score(xgb, X, y, cv=5, scoring='neg_mean_absolute_error')
        grid_results[i, j] = -scores.mean()

# Display as DataFrame
grid_df = pd.DataFrame(grid_results, index=learning_rates_grid, columns=n_estimators_grid)
grid_df.index.name = 'learning_rate'
grid_df.columns.name = 'n_estimators'
print(grid_df.round(0).to_string())

# Best combination
best_idx = np.unravel_index(np.argmin(grid_results), grid_results.shape)
best_lr = learning_rates_grid[best_idx[0]]
best_ne = n_estimators_grid[best_idx[1]]
best_mae = grid_results[best_idx]

print(f"\nBest: learning_rate={best_lr}, n_estimators={best_ne}")
print(f"  MAE: ${best_mae:,.2f}")
print(f"  Improvement over baseline: ${baseline_mae - best_mae:,.2f} ({100*(baseline_mae - best_mae)/baseline_mae:.1f}%)")

---
# 5 Feature Importance & Final Evaluation

#### We train the best model on the full dataset to extract feature importances, then do a proper train/test split to report final out-of-sample MAE.

In [ ]:
# Feature importances from best model
best_model = XGBRegressor(learning_rate=best_lr, n_estimators=best_ne,
                          random_state=42, n_jobs=-1, verbosity=0)
best_model.fit(X, y)

importances = best_model.feature_importances_
for feat, imp in sorted(zip(feature_cols, importances), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.3f}")

In [ ]:
# Train/test split for final evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

best_model_split = XGBRegressor(learning_rate=best_lr, n_estimators=best_ne,
                                random_state=42, n_jobs=-1, verbosity=0)
best_model_split.fit(X_train, y_train)
y_pred = best_model_split.predict(X_test)

test_mae = mean_absolute_error(y_test, y_pred)
print(f"Test MAE: ${test_mae:,.2f}")
print(f"Trained on {len(X_train)} samples, tested on {len(X_test)} samples")